In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
  mean_absolute_error,
  mean_squared_error, 
  root_mean_squared_error, 
  r2_score,accuracy_score,
  precision_score,
  recall_score,
  f1_score,
  confusion_matrix,
  classification_report,
  roc_auc_score)
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_regression, chi2

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [3]:
df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("Test.csv")
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [4]:
print(df_train.head())
print("***************************************************")
print(df_train.info())
print("***************************************************")
print(df_train.describe())

   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   

  LandContour Utilities LotConfig LandSlope Neighborhood Condition1  \
0         Lvl    AllPub    Inside       Gtl      CollgCr       Norm   
1         Lvl    AllPub       FR2       Gtl      Veenker      Feedr   
2         Lvl    AllPub    Inside       Gtl      CollgCr       Norm   
3         Lvl    AllPub    Corner       Gtl      Crawfor       Norm   
4         Lvl    AllPub       FR2       Gtl      NoRidge       Norm   

  Condition2 BldgType HouseStyle  OverallQual  OverallCond  YearBuilt  \
0       Norm     1Fam     2Story            7          

In [5]:
print("Is Null:")
fields_with_null_values = df_train.isnull().sum()
fields_with_null_values = fields_with_null_values[fields_with_null_values > 0]

null_info = pd.DataFrame({
    'Missing Values': fields_with_null_values,
    'Data Type': df_train.dtypes[fields_with_null_values.index]
})

print(null_info)

Is Null:
              Missing Values Data Type
LotFrontage              259   float64
Alley                   1369       str
MasVnrType               872       str
MasVnrArea                 8   float64
BsmtQual                  37       str
BsmtCond                  37       str
BsmtExposure              38       str
BsmtFinType1              37       str
BsmtFinType2              38       str
Electrical                 1       str
FireplaceQu              690       str
GarageType                81       str
GarageYrBlt               81   float64
GarageFinish              81       str
GarageQual                81       str
GarageCond                81       str
PoolQC                  1453       str
Fence                   1179       str
MiscFeature             1406       str


In [6]:
df_train["LotFrontage"] = (
    df_train.groupby("Neighborhood")["LotFrontage"]
      .transform(lambda x: x.fillna(x.median()))
)

df_train["Electrical"] = df_train["Electrical"].fillna(
    df_train["Electrical"].mode()[0]
)

df_train["Alley"] = df_train["Alley"].fillna("None")
df_train["MasVnrType"] = df_train["MasVnrType"].fillna("None")
df_train["MasVnrArea"] = df_train["MasVnrArea"].fillna(0)
df_train["BsmtQual"] = df_train["BsmtQual"].fillna("None")
df_train["BsmtCond"] = df_train["BsmtCond"].fillna("None")
df_train["BsmtExposure"] = df_train["BsmtExposure"].fillna("None")
df_train["BsmtFinType1"] = df_train["BsmtFinType1"].fillna("None")
df_train["BsmtFinType2"] = df_train["BsmtFinType2"].fillna("None")
df_train["FireplaceQu"] = df_train["FireplaceQu"].fillna("None")
df_train["GarageType"] = df_train["GarageType"].fillna("None")
df_train["GarageYrBlt"] = df_train["GarageYrBlt"].fillna(0)
df_train["GarageFinish"] = df_train["GarageFinish"].fillna("None")
df_train["GarageQual"] = df_train["GarageQual"].fillna("None")
df_train["GarageCond"] = df_train["GarageCond"].fillna("None")
df_train["PoolQC"] = df_train["PoolQC"].fillna("None")
df_train["Fence"] = df_train["Fence"].fillna("None")
df_train["MiscFeature"] = df_train["MiscFeature"].fillna("None")

In [7]:
print("Is Null:")
fields_with_null_values = df_test.isnull().sum()
fields_with_null_values = fields_with_null_values[fields_with_null_values > 0]

null_info = pd.DataFrame({
    'Missing Values': fields_with_null_values,
    'Data Type': df_test.dtypes[fields_with_null_values.index]
})

print(null_info)

Is Null:
              Missing Values Data Type
MSZoning                   4       str
LotFrontage              227   float64
Alley                   1352       str
Utilities                  2       str
Exterior1st                1       str
Exterior2nd                1       str
MasVnrType               894       str
MasVnrArea                15   float64
BsmtQual                  44       str
BsmtCond                  45       str
BsmtExposure              44       str
BsmtFinType1              42       str
BsmtFinSF1                 1   float64
BsmtFinType2              42       str
BsmtFinSF2                 1   float64
BsmtUnfSF                  1   float64
TotalBsmtSF                1   float64
BsmtFullBath               2   float64
BsmtHalfBath               2   float64
KitchenQual                1       str
Functional                 2       str
FireplaceQu              730       str
GarageType                76       str
GarageYrBlt               78   float64
GarageFinish    

In [8]:
neighborhood_avg = [
  "LotFrontage"
]

str_category = [
  "MSZoning",
  "Utilities",
  "Exterior1st",
  "Exterior2nd",
  "SaleType"
]

na_to_none = [
  "Alley",
  "MasVnrType",
  "BsmtQual",
  "BsmtCond",
  "BsmtExposure",
  "BsmtFinType1",
  "BsmtFinType2",
  "KitchenQual",
  "Functional",
  "FireplaceQu",
  "GarageType",
  "GarageFinish",
  "GarageQual",
  "GarageCond",
  "PoolQC",
  "Fence",
  "MiscFeature"
]

na_to_0 = [
  "BsmtFinSF1",
  "BsmtFinSF2",
  "BsmtUnfSF",
  "TotalBsmtSF",
  "BsmtFullBath",
  "BsmtHalfBath",
  "MasVnrArea",
  "GarageYrBlt",
  "GarageCars",
  "GarageArea"
]

for item in neighborhood_avg:
  df_test[item] = (
    df_test.groupby("Neighborhood")[item]
    .transform(lambda x: x.fillna(x.median()))
  )

for item in str_category:
  df_test[item] = df_test[item].fillna(
      df_test[item].mode()[0]
  )

for item in na_to_none:
  df_test[item] = df_test[item].fillna("None")

for item in na_to_0:
  df_test[item] = df_test[item].fillna(0)

In [9]:
# Creating Classification Dataset
classification_df_train = df_train.copy()

# 0 = Low, 1 = Medium, 2 = High
classification_df_train["PriceCategory"] = pd.qcut(
    classification_df_train["SalePrice"],
    q=3,
    labels=[0, 1, 2]
)

classification_df_train.drop(
    columns=["SalePrice"],
    inplace=True
)
classification_df_train["PriceCategory"].value_counts()


PriceCategory
1    490
0    487
2    483
Name: count, dtype: int64

Nominal Catagories:
"MSSubClass",
"MSZoning",
"Street",
"Alley",
"LotShape",
"LandContour",
"Utilities",
"LotConfig",
"LandSlope",
"Neighborhood",
"Condition1",
"Condition2",
"BldgType",
"HouseStyle",
"RoofStyle",
"RoofMatl",
"Exterior1st",
"Exterior2nd",
"MasVnrType",
"Foundation",
"Heating",
"Electrical",
"GarageType",
"Fence",
"MiscFeature",
"SaleType",
"SaleCondition"

Y/N:
"CentralAir"

Ordinal 1-10:
"OverallQual",
"OverallCond"

Ordinal Po-Ex(5):
"ExterQual",
"ExterCond",
"HeatingQC",
"KitchenQual"

Ordinal Na-Ex(6):
"BsmtQual",
"BsmtCond",
"FireplaceQu",
"GarageQual",
"GarageCond"

Ordinal Na-Ex(5):
"PoolQC"

Ordinal Misc:
"BsmtExposure",
"BsmtFinType1",
"BsmtFinType2",
"Functional",
"GarageFinish",
"PavedDrive"??



In [10]:
#Ordinal Maps
datasets = [
  df_train,
  classification_df_train,
  df_test
]

for dataset in datasets:
  # Yes/No Map
  quality_map = {
    "N": 0,
    "Y": 1
  }

  dataset["CentralAir"] = dataset["CentralAir"].map(quality_map)

  # Poor - Excellent 5 Rankings Map
  quality_map = {
    "Po": 0,
    "NA": 0,
    "None": 0,
    "Fa": 1,
    "TA": 2,
    "Gd": 3,
    "Ex": 4
  }

  ordinal_po_ex_5 = [
  "ExterQual",
  "ExterCond",
  "HeatingQC",
  "KitchenQual"
  ]

  for category in ordinal_po_ex_5:
    dataset[category]=dataset[category].map(quality_map)

  # NA - Excellent 6 Rankings Map
  quality_map = {
    "NA": 0,
    "None": 0,
    "Po": 1,
    "Fa": 2,
    "TA": 3,
    "Gd": 4,
    "Ex": 5
  }

  ordinal_na_ex_6 = [
  "BsmtQual",
  "BsmtCond",
  "FireplaceQu",
  "GarageQual",
  "GarageCond"
  ]

  for category in ordinal_na_ex_6:
    dataset[category]=dataset[category].map(quality_map)

  # NA - Excellent 5 Rankings Map
  quality_map = {
    "NA": 0,
    "None": 0,
    "Fa": 1,
    "TA": 2,
    "Gd": 3,
    "Ex": 4
  }

  ordinal_na_ex_5 = [
    "PoolQC"
  ]

  for category in ordinal_na_ex_5:
    dataset[category]=dataset[category].map(quality_map)


  # Misc. Ordinal Rankings Map

  # "BsmtExposure"
  quality_map ={
    "NA": 0,
    "None": 0,
    "No": 0,
    "Mn":	2,
    "Av":	3,
    "Gd":	4
  }

  dataset["BsmtExposure"]=dataset["BsmtExposure"].map(quality_map)

  # "BsmtFinType1"
  # "BsmtFinType2"
  quality_map = {
    "NA": 0,
    "None": 0,
    "No": 0,
    "Unf": 1,
    "LwQ": 2,
    "Rec": 3,
    "BLQ": 4,
    "ALQ": 5,
    "GLQ": 6
  }

  dataset["BsmtFinType1"]=dataset["BsmtFinType1"].map(quality_map)
  dataset["BsmtFinType2"]=dataset["BsmtFinType2"].map(quality_map)

  # "Functional"
  quality_map ={
    "NA": 0,
    "None": 0,
    "Sal": 0,
    "Sev": 1,
    "Maj2": 2,
    "Maj1": 3,
    "Mod": 4,
    "Min2": 5,
    "Min1": 6,
    "Typ": 7
  }
  dataset["Functional"]=dataset["Functional"].map(quality_map)

  # "GarageFinish"
  quality_map ={
    "NA": 0,
    "None": 0,
    "Unf": 1,
    "RFn": 2,
    "Fin": 3
  }

  dataset["GarageFinish"]=dataset["GarageFinish"].map(quality_map)


  # "PavedDrive"
  quality_map ={
    "N": 0,
    "NA": 0,
    "None": 0,
    "P" : 1,
    "Y": 2
  }

  dataset["PavedDrive"]=dataset["PavedDrive"].map(quality_map)

In [11]:
# Regression dataset targeting sale price
X = df_train.drop(columns=["SalePrice"])
y=df_train["SalePrice"]



# Classification dataset targeting price category 
y_class = classification_df_train["PriceCategory"]
X_class = classification_df_train.drop(columns=["PriceCategory"])

X_test = df_test

In [12]:
#Nominal Maps
nominal_columns = [
"MSSubClass",
"MSZoning",
"Street",
"Alley",
"LotShape",
"LandContour",
"Utilities",
"LotConfig",
"LandSlope",
"Neighborhood",
"Condition1",
"Condition2",
"BldgType",
"HouseStyle",
"RoofStyle",
"RoofMatl",
"Exterior1st",
"Exterior2nd",
"MasVnrType",
"Foundation",
"Heating",
"Electrical",
"GarageType",
"Fence",
"MiscFeature",
"SaleType",
"SaleCondition"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat",
        OneHotEncoder(handle_unknown="ignore", drop="first"),
        nominal_columns)
    ],
    remainder="passthrough"
)

X = preprocessor.fit_transform(X)
X_class = preprocessor.transform(X_class)
X_test = preprocessor.transform(X_test)

c:\code\d789_task2\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [13]:
# Regression Split
X_train, X_val, y_train, y_val = train_test_split(
  X,
  y,
  test_size=0.2,
  random_state=42
)

# Classification Split
X_class_train, X_class_val, y_class_train, y_class_val = train_test_split(
  X_class,
  y_class,
  test_size=0.2,
  random_state=42
)

# Create Results List
results = []

In [14]:
# Chi-Square Feature Selection - Classification Dataset - (Running before scaling to avoid negatives)
selector = SelectKBest(
  score_func=chi2,
  k=20
)

X_class_train_chi2 = selector.fit_transform(X_class_train, y_class_train)
X_class_val_chi2 = selector.transform(X_class_val)

In [15]:
# Scale regression dataset
reg_scaler = StandardScaler()
X_train = reg_scaler.fit_transform(X_train)
X_val = reg_scaler.transform(X_val)

# Scale classification dataset
class_scaler = StandardScaler()
X_class_train = class_scaler.fit_transform(X_class_train)
X_class_val = class_scaler.transform(X_class_val)

# Scale Chi-Square Classification Dataset
chi2_scaler = StandardScaler()
X_class_train_chi2 = chi2_scaler.fit_transform(X_class_train_chi2)
X_class_val_chi2 = chi2_scaler.transform(X_class_val_chi2)


# Fix classification dataset label type
y_class_train = y_class_train.astype("int32")
y_class_val = y_class_val.astype("int32")

In [16]:
# PCA Feature Selection - Regression DataSet
pca = PCA(n_components=0.95)

X_train_PCA = pca.fit_transform(X_train)
X_val_PCA = pca.transform(X_val)


In [17]:
# ANOVA F-Test Feature Selection - Regression & Classification Datasets
selector = SelectKBest(
  score_func=f_regression,
  k=20
)

X_train_ANOVA = selector.fit_transform(X_train, y_train)
X_val_ANOVA = selector.transform(X_val)

X_class_train_ANOVA = selector.fit_transform(X_class_train, y_class_train)
X_class_val_ANOVA = selector.transform(X_class_val)

In [18]:
# Regression Model - Linear Regression
model = LinearRegression()

# Full feature set
model.fit(X_train, y_train)

y_pred = model.predict(X_val)

mae = mean_absolute_error(
    y_val, 
    y_pred
)

mse = mean_squared_error(
    y_val, 
    y_pred
)

rmse = root_mean_squared_error(
    y_val,
    y_pred,
)

r2 = r2_score(
    y_val,
    y_pred
)

results.append({
    "Model Type": "Classical Regression",
    "Model": "Linear Regression",
    "Feature Set": "Full Feature Set",
    "Output": "Predicted SalePrice",
    "MAE": mae,
    "MSE": mse,
    "RMSE": rmse,
    "R2": r2,
    "Accuracy": None,
    "Precision": None,
    "Recall": None,
    "F1": None,
    "AUC": None
})  


# PCA Selected Feature Set
model.fit(X_train_PCA, y_train)

y_pred = model.predict(X_val_PCA)

mae = mean_absolute_error(
    y_val, 
    y_pred
)

mse = mean_squared_error(
    y_val, 
    y_pred
)

rmse = root_mean_squared_error(
    y_val,
    y_pred,
)

r2 = r2_score(
    y_val,
    y_pred
)

results.append({
    "Model Type": "Classical Regression",
    "Model": "Linear Regression",
    "Feature Set": "PCA Selected Feature Set",
    "Output": "Predicted SalePrice",
    "MAE": mae,
    "MSE": mse,
    "RMSE": rmse,
    "R2": r2,
    "Accuracy": None,
    "Precision": None,
    "Recall": None,
    "F1": None,
    "AUC": None
})  

# ANOVA Selected Feature Set
model.fit(X_train_ANOVA, y_train)

y_pred = model.predict(X_val_ANOVA)

mae = mean_absolute_error(
    y_val, 
    y_pred
)

mse = mean_squared_error(
    y_val, 
    y_pred
)

rmse = root_mean_squared_error(
    y_val,
    y_pred,
)

r2 = r2_score(
    y_val,
    y_pred
)

results.append({
    "Model Type": "Classical Regression",
    "Model": "Linear Regression",
    "Feature Set": "ANOVA Selected Feature Set",
    "Output": "Predicted SalePrice",
    "MAE": mae,
    "MSE": mse,
    "RMSE": rmse,
    "R2": r2,
    "Accuracy": None,
    "Precision": None,
    "Recall": None,
    "F1": None,
    "AUC": None
})  

In [19]:
# Classical Classification Model - Logistical Regression
model = LogisticRegression(
    max_iter=5000,
    random_state=42
)

model.fit(X_class_train, y_class_train)

y_pred = model.predict(X_class_val)
y_proba = model.predict_proba(X_class_val)

accuracy = accuracy_score(y_class_val, y_pred)
precision = precision_score(y_class_val, y_pred, average="weighted")
recall = recall_score(y_class_val, y_pred, average="weighted")
f1 = f1_score(y_class_val, y_pred, average="weighted")

auc = roc_auc_score(
    y_class_val,
    y_proba,
    multi_class="ovr",
    average="weighted"
)

cm = confusion_matrix(y_class_val, y_pred)

print("Confusion Matrix:")
print(cm)

print("Classification Report:")
print(classification_report(y_class_val, y_pred))

results.append({
    "Model Type": "Classical Classification",
    "Model": "Logistic Regression",
    "Feature Set": "Full Feature Set",
    "Output": "Predicted PriceCategory",
    "MAE": None,
    "MSE": None,
    "RMSE": None,
    "R2": None,
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "AUC": auc
})

# ANOVA Selected Feature Set

model.fit(X_class_train_ANOVA, y_class_train)

y_pred = model.predict(X_class_val_ANOVA)
y_proba = model.predict_proba(X_class_val_ANOVA)

accuracy = accuracy_score(y_class_val, y_pred)
precision = precision_score(y_class_val, y_pred, average="weighted")
recall = recall_score(y_class_val, y_pred, average="weighted")
f1 = f1_score(y_class_val, y_pred, average="weighted")

auc = roc_auc_score(
    y_class_val,
    y_proba,
    multi_class="ovr",
    average="weighted"
)

cm = confusion_matrix(y_class_val, y_pred)

print("Confusion Matrix:")
print(cm)

print("Classification Report:")
print(classification_report(y_class_val, y_pred))

results.append({
    "Model Type": "Classical Classification",
    "Model": "Logistic Regression",
    "Feature Set": "ANOVA Selected Feature Set",
    "Output": "Predicted PriceCategory",
    "MAE": None,
    "MSE": None,
    "RMSE": None,
    "R2": None,
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "AUC": auc
})

# Chi-Squared Selected Feature Set
model.fit(X_class_train_chi2, y_class_train)

y_pred = model.predict(X_class_val_chi2)
y_proba = model.predict_proba(X_class_val_chi2)

accuracy = accuracy_score(y_class_val, y_pred)
precision = precision_score(y_class_val, y_pred, average="weighted")
recall = recall_score(y_class_val, y_pred, average="weighted")
f1 = f1_score(y_class_val, y_pred, average="weighted")

auc = roc_auc_score(
    y_class_val,
    y_proba,
    multi_class="ovr",
    average="weighted"
)

cm = confusion_matrix(y_class_val, y_pred)

print("Confusion Matrix:")
print(cm)

print("Classification Report:")
print(classification_report(y_class_val, y_pred))

results.append({
    "Model Type": "Classical Classification",
    "Model": "Logistic Regression",
    "Feature Set": "Chi Squared Selected Feature Set",
    "Output": "Predicted PriceCategory",
    "MAE": None,
    "MSE": None,
    "RMSE": None,
    "R2": None,
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "AUC": auc
})

Confusion Matrix:
[[94 16  0]
 [16 61 11]
 [ 0 16 78]]
Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.85      0.85       110
           1       0.66      0.69      0.67        88
           2       0.88      0.83      0.85        94

    accuracy                           0.80       292
   macro avg       0.80      0.79      0.79       292
weighted avg       0.80      0.80      0.80       292

Confusion Matrix:
[[99 11  0]
 [16 61 11]
 [ 0  9 85]]
Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.90      0.88       110
           1       0.75      0.69      0.72        88
           2       0.89      0.90      0.89        94

    accuracy                           0.84       292
   macro avg       0.83      0.83      0.83       292
weighted avg       0.84      0.84      0.84       292

Confusion Matrix:
[[93 15  2]
 [17 65  6]
 [ 0 23 71]]
Classification Report:
    

In [ ]:
# Deep Learning Regression Model - Feedforward Dense Neural Network - Full Feature Set
regressor = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(1)
])

regressor.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

history = regressor.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

y_pred = regressor.predict(X_val).flatten()

mae = mean_absolute_error(
    y_val, 
    y_pred
)

mse = mean_squared_error(
    y_val, 
    y_pred
)

rmse = root_mean_squared_error(
    y_val,
    y_pred,
)

r2 = r2_score(
    y_val,
    y_pred
)

results.append({
    "Model Type": "Deep Learning Regression",
    "Model": "Dense Neural Network Regression",
    "Feature Set": "Full Feature Set",
    "Output": "Predicted SalePrice",
    "MAE": mae,
    "MSE": mse,
    "RMSE": rmse,
    "R2": r2,
    "Accuracy": None,
    "Precision": None,
    "Recall": None,
    "F1": None,
    "AUC": None
})  


# Deep Learning Regression Model - Feedforward Dense Neural Network - PCA Selected Feature Set
regressor = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_PCA.shape[1],)),
    Dense(64, activation='relu'),
    Dense(1)
])

regressor.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

history = regressor.fit(
    X_train_PCA,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

y_pred = regressor.predict(X_val_PCA).flatten()

mae = mean_absolute_error(
    y_val, 
    y_pred
)

mse = mean_squared_error(
    y_val, 
    y_pred
)

rmse = root_mean_squared_error(
    y_val,
    y_pred,
)

r2 = r2_score(
    y_val,
    y_pred
)

results.append({
    "Model Type": "Deep Learning Regression",
    "Model": "Dense Neural Network Regression",
    "Feature Set": "PCA Selected Feature Set",
    "Output": "Predicted SalePrice",
    "MAE": mae,
    "MSE": mse,
    "RMSE": rmse,
    "R2": r2,
    "Accuracy": None,
    "Precision": None,
    "Recall": None,
    "F1": None,
    "AUC": None
})  

# Deep Learning Regression Model - Feedforward Dense Neural Network - ANOVA Selected Feature Set
regressor = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_ANOVA.shape[1],)),
    Dense(64, activation='relu'),
    Dense(1)
])

regressor.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

history = regressor.fit(
    X_train_ANOVA,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

y_pred = regressor.predict(X_val_ANOVA).flatten()

mae = mean_absolute_error(
    y_val, 
    y_pred
)

mse = mean_squared_error(
    y_val, 
    y_pred
)

rmse = root_mean_squared_error(
    y_val,
    y_pred,
)

r2 = r2_score(
    y_val,
    y_pred
)

results.append({
    "Model Type": "Deep Learning Regression",
    "Model": "Dense Neural Network Regression",
    "Feature Set": "ANOVA Selected Feature Set",
    "Output": "Predicted SalePrice",
    "MAE": mae,
    "MSE": mse,
    "RMSE": rmse,
    "R2": r2,
    "Accuracy": None,
    "Precision": None,
    "Recall": None,
    "F1": None,
    "AUC": None
}) 

c:\code\d789_task2\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


KeyboardInterrupt: 

In [ ]:
# Deep Learning Classification - Dense Neural Network Classification - Full Feature Set
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_class_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_class_train,
    y_class_train.astype("int32"),
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

y_proba = model.predict(X_class_val)
y_pred = np.argmax(y_proba, axis=1)

accuracy = accuracy_score(y_class_val, y_pred)
precision = precision_score(y_class_val, y_pred, average="weighted")
recall = recall_score(y_class_val, y_pred, average="weighted")
f1 = f1_score(y_class_val, y_pred, average="weighted")

auc = roc_auc_score(
    y_class_val,
    y_proba,
    multi_class="ovr",
    average="weighted"
)

cm = confusion_matrix(y_class_val, y_pred)

print("Confusion Matrix:")
print(cm)

print("Classification Report:")
print(classification_report(y_class_val, y_pred))

results.append({
    "Model Type": "Deep Learning Classification",
    "Model": "Dense Neural Network Classification",
    "Feature Set": "Full Feature Set",
    "Output": "Predicted PriceCategory",
    "MAE": None,
    "MSE": None,
    "RMSE": None,
    "R2": None,
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "AUC": auc
})

# Deep Learning Classification - Dense Neural Network Classification - ANOVA Selected Feature Set
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_class_train_ANOVA.shape[1],)),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_class_train_ANOVA,
    y_class_train.astype("int32"),
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

y_proba = model.predict(X_class_val_ANOVA)
y_pred = np.argmax(y_proba, axis=1)

accuracy = accuracy_score(y_class_val, y_pred)
precision = precision_score(y_class_val, y_pred, average="weighted")
recall = recall_score(y_class_val, y_pred, average="weighted")
f1 = f1_score(y_class_val, y_pred, average="weighted")

auc = roc_auc_score(
    y_class_val,
    y_proba,
    multi_class="ovr",
    average="weighted"
)

cm = confusion_matrix(y_class_val, y_pred)

print("Confusion Matrix:")
print(cm)

print("Classification Report:")
print(classification_report(y_class_val, y_pred))

results.append({
    "Model Type": "Deep Learning Classification",
    "Model": "Dense Neural Network Classification",
    "Feature Set": "ANOVA Selected Feature Set",
    "Output": "Predicted PriceCategory",
    "MAE": None,
    "MSE": None,
    "RMSE": None,
    "R2": None,
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "AUC": auc
})

# Deep Learning Classification - Dense Neural Network Classification - Chi-Squared Selected Feature Set
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_class_train_chi2.shape[1],)),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_class_train_chi2,
    y_class_train.astype("int32"),
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

y_proba = model.predict(X_class_val_chi2)
y_pred = np.argmax(y_proba, axis=1)

accuracy = accuracy_score(y_class_val, y_pred)
precision = precision_score(y_class_val, y_pred, average="weighted")
recall = recall_score(y_class_val, y_pred, average="weighted")
f1 = f1_score(y_class_val, y_pred, average="weighted")

auc = roc_auc_score(
    y_class_val,
    y_proba,
    multi_class="ovr",
    average="weighted"
)

cm = confusion_matrix(y_class_val, y_pred)

print("Confusion Matrix:")
print(cm)

print("Classification Report:")
print(classification_report(y_class_val, y_pred))

results.append({
    "Model Type": "Deep Learning Classification",
    "Model": "Dense Neural Network Classification",
    "Feature Set": "Chi-Squared Feature Set",
    "Output": "Predicted PriceCategory",
    "MAE": None,
    "MSE": None,
    "RMSE": None,
    "R2": None,
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "AUC": auc
})

Epoch 1/100


c:\code\d789_task2\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6178 - loss: 0.8098 - val_accuracy: 0.7564 - val_loss: 0.5923
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8373 - loss: 0.4271 - val_accuracy: 0.7821 - val_loss: 0.4976
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8972 - loss: 0.3051 - val_accuracy: 0.7821 - val_loss: 0.4822
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9347 - loss: 0.2240 - val_accuracy: 0.8333 - val_loss: 0.4540
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9529 - loss: 0.1670 - val_accuracy: 0.8077 - val_loss: 0.4688
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9743 - loss: 0.1281 - val_accuracy: 0.8162 - val_loss: 0.4780
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9818 - loss: 0.0984 - val_accuracy: 0.8120 - val_loss: 0.5025
Epoch 8/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9882 - loss: 0.0773 - val_accuracy: 0.8120 - val_loss: 0.5

c:\code\d789_task2\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7227 - loss: 0.6628 - val_accuracy: 0.7949 - val_loss: 0.5205
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7880 - loss: 0.4993 - val_accuracy: 0.8077 - val_loss: 0.4681
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8244 - loss: 0.4529 - val_accuracy: 0.8333 - val_loss: 0.4296
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8340 - loss: 0.4271 - val_accuracy: 0.8248 - val_loss: 0.4247
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8405 - loss: 0.3986 - val_accuracy: 0.8333 - val_loss: 0.4041
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8512 - loss: 0.3854 - val_accuracy: 0.8333 - val_loss: 0.4039
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8597 - loss: 0.3644 - val_accuracy: 0.8376 - val_loss: 0.3956
Epoch 8/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8640 - loss: 0.3438 - val_accuracy: 0.8462 - val_loss: 0.3

c:\code\d789_task2\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5300 - loss: 0.9251 - val_accuracy: 0.6282 - val_loss: 0.7486
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6734 - loss: 0.7008 - val_accuracy: 0.7222 - val_loss: 0.6458
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7173 - loss: 0.6247 - val_accuracy: 0.7350 - val_loss: 0.6156
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7452 - loss: 0.5827 - val_accuracy: 0.7436 - val_loss: 0.6157
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7687 - loss: 0.5516 - val_accuracy: 0.7393 - val_loss: 0.5946
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7709 - loss: 0.5287 - val_accuracy: 0.7393 - val_loss: 0.5943
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7773 - loss: 0.5088 - val_accuracy: 0.7393 - val_loss: 0.5970
Epoch 8/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7901 - loss: 0.4891 - val_accuracy: 0.7564 - val_loss: 0.5

In [ ]:
results_df = pd.DataFrame(results)
results_df

,Model Type,Model,Feature Set,Output,MAE,MSE,RMSE,R2,Accuracy,Precision,Recall,F1,AUC
0,Classical Regression,Linear Regression,Full Feature Set,Predicted SalePrice,22096.547273,2.645353e+09,51432.999833,0.655119,NaN,NaN,NaN,NaN,NaN
1,Classical Regression,Linear Regression,PCA Selected Feature Set,Predicted SalePrice,21709.602454,1.173454e+09,34255.719228,0.847014,NaN,NaN,NaN,NaN,NaN
2,Classical Regression,Linear Regression,ANOVA Selected Feature Set,Predicted SalePrice,22908.699679,1.324397e+09,36392.270671,0.827335,NaN,NaN,NaN,NaN,NaN
3,Classical Classification,Logistic Regression,Full Feature Set,Predicted PriceCategory,NaN,NaN,NaN,NaN,0.797945,0.801721,0.797945,0.799473,0.931262
4,Classical Classification,Logistic Regression,ANOVA Selected Feature Set,Predicted PriceCategory,NaN,NaN,NaN,NaN,0.839041,0.836289,0.839041,0.837096,0.954953
5,Classical Classification,Logistic Regression,Chi Squared Selected Feature Set,Predicted PriceCategory,NaN,NaN,NaN,NaN,0.784247,0.797997,0.784247,0.787847,0.932527
6,Deep Learning Regression,Dense Neural Network Regression,Full Feature Set,Predicted SalePrice,29975.757812,2.034875e+09,45109.585938,0.734708,NaN,NaN,NaN,NaN,NaN
7,Deep Learning Regression,Dense Neural Network Regression,PCA Selected Feature Set,Predicted SalePrice,25540.015625,1.339621e+09,36600.832031,0.825350,NaN,NaN,NaN,NaN,NaN
8,Deep Learning Regression,Dense Neural Network Regression,ANOVA Selected Feature Set,Predicted SalePrice,46253.707031,3.515444e+09,59291.179688,0.541683,NaN,NaN,NaN,NaN,NaN
9,Deep Learning Classification,Dense Neural Network Classification,Full Feature Set,Predicted PriceCategory,NaN,NaN,NaN,NaN,0.791096,0.793724,0.791096,0.791555,0.939301
